In [ ]:
# Necessary import
import os
import uuid
import pandas as pd
import mlflow

In [ ]:
# Data parameters values
year = 2021
month = 2
taxi_type = 'green'

# Input data address
input_file = f'https://s3.amazonaws.com/nyc-tlc/trip+data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'
# Address of the output
output_file = f'output/{taxi_type}/{year:04d}-{month:02d}.parquet'

# Get the run ID
RUN_ID = os.getenv('RUN_ID', '1ca05c6d23f44066a4a4dcdbe1639de4')

In [ ]:
# Function to generate ride IDs
def generate_uuids(n):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))
    return ride_ids

# Function for reading the data
def read_dataframe(filename: str):
    # read the parquet file
    df = pd.read_parquet(filename)

    # Feature engineering to create a duration column
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    # Convert durations to minutes
    df.duration = df.duration.dt.total_seconds() / 60
    # Filter durations
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    # Set the ride ID
    df['ride_id'] = generate_uuids(len(df))

    return df # to return the prepared dataframe

# Function for building data dictionaries
def prepare_dictionaries(df: pd.DataFrame):
    # Set of categorical features
    categorical = ['PULocationID', 'DOLocationID']
    # Convert categorical features to string
    df[categorical] = df[categorical].astype(str)
    
    # Create a trajet feature
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    # Set of features
    categorical = ['PU_DO'] # categorical
    numerical = ['trip_distance'] # numerical
    # Build the data dictionaries
    dicts = df[categorical + numerical].to_dict(orient = 'records')
    return dicts # to return dictionaries

In [9]:
def load_model(run_id):
    logged_model = f's3://mlflow-models-alexey/1/{RUN_ID}/artifacts/model'
    model = mlflow.pyfunc.load_model(logged_model)
    return model


def apply_model(input_file, run_id, output_file):

    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)

    
    model = load_model(run_id)
    y_pred = model.predict(dicts)

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['duration']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id
    
    df_result.to_parquet(output_file, index=False)

In [11]:
apply_model(input_file=input_file, run_id=RUN_ID, output_file=output_file)

In [12]:
!ls output/green/

2021-02.parquet  2021-03.parquet


---